# 關鍵詞辨識展示：輸入一句手語的 .npy，輸出字幕中的幾個詞

使用 `sign_keyword_ctc.ipynb` 訓練好的 `CTC` 模型。模型對 100 個常見詞各給一個分數，
**只輸出分數超過門檻的詞** (最多 `MAX_WORDS` 個)；沒有任何詞超過門檻就回答「無法判斷」。

步驟：
1. **挑門檻**：在**驗證集**上找出讓「輸出的詞有 `TARGET_PRECISION` 是對的」的門檻 (測試集不參與)
2. **預期效果**：把門檻套到**測試集**，算出實際的準確率、會回答的句子比例
3. **存設定檔** `keyword_ctc/demo_config.json`，之後 `keyword_predict.py` 都用它
4. **實際示範**：讀 `.npy` → 輸出關鍵詞，並對照字幕

**準確率的定義**：模型輸出的詞裡，有多少比例真的出現在字幕中 (精確率)。
測試集通常比驗證集低幾個百分點，所以 `TARGET_PRECISION` 預設 0.7，讓測試集維持在 60% 以上。

**執行前**：要先跑完 `sign_keyword_ctc.ipynb` (需要 `keyword_ctc/CTC/best.pt`、`scores.npz`、`keyword_ctc/keywords.json`)。

In [ ]:
import csv
import json
import logging
import re
from pathlib import Path

import jieba
import numpy as np

# ================================================================ 路徑設定
MODEL_DIR = Path(r"keyword_ctc")             # sign_keyword_ctc.ipynb 的輸出
VARIANT = "CTC"                              # 用哪個版本的模型
DATA_DIR = Path(r"data")
SEG_DIR = Path(r"..\Data Preparation\Phase5-Pause\output\03segments_face")
JIEBA_DICT = DATA_DIR / "dict.txt.big"

# ================================================================ 展示設定
TARGET_PRECISION = 0.7     # 在驗證集上要求「輸出的詞有多少比例是對的」
MAX_WORDS = 5              # 每句最多輸出幾個詞

# ================================================================ 模型設定 (必須跟 sign_keyword_ctc.ipynb 訓練時相同)
FEAT_DIM = 292
USE_FACE = False
FRAME_STRIDE = 2
MAX_FRAMES = 300
NORM_FILE = DATA_DIR / "norm_masked_face.npz"
MODEL = {"d_model": 256, "n_heads": 4, "enc_layers": 4, "ff_dim": 1024, "dropout": 0.3}
MIN_WORD_LEN = 2           # 跟訓練時的斷詞規則相同

jieba.setLogLevel(logging.WARNING)
if JIEBA_DICT.exists():
    jieba.set_dictionary(str(JIEBA_DICT))
jieba.initialize()
WORD_RE = re.compile(r"[一-鿿]+")

KEYWORDS = list(json.loads((MODEL_DIR / "keywords.json").read_text(encoding="utf-8")))
print(f"{len(KEYWORDS)} 個關鍵詞：{'、'.join(KEYWORDS[:30])}…")


def words_of(text):
    return {w for w in jieba.lcut(text) if len(w) >= MIN_WORD_LEN and WORD_RE.fullmatch(w)}


def read_rows(split):
    with (DATA_DIR / f"{split}.csv").open(encoding="utf-8-sig", newline="") as f:
        return [(row["path"], row["text"]) for row in csv.DictReader(f)]


def label_matrix(rows):
    return np.array([[w in words_of(t) for w in KEYWORDS] for _, t in rows], dtype=np.float32)


val_rows, test_rows, train_rows = read_rows("val"), read_rows("test"), read_rows("train")
val_labels, test_labels = label_matrix(val_rows), label_matrix(test_rows)
scores = np.load(MODEL_DIR / VARIANT / "scores.npz")
val_scores = (scores["val"] + scores["val_ctc"]) / 2          # 分類層與 CTC 偵測分數的平均
test_scores = (scores["test"] + scores["test_ctc"]) / 2
assert val_scores.shape[1] == len(KEYWORDS), "scores.npz 的詞數跟 keywords.json 對不起來"

## 1. 挑門檻與預期效果
輸出規則：分數 ≥ 門檻的詞，依分數排序取前 `MAX_WORDS` 個。

| 指標 | 意思 |
|---|---|
| **精確率** | 輸出的詞裡，真的出現在字幕中的比例 (**目標 ≥ 60%**) |
| 有回答的句子 | 至少輸出 1 個詞的句子比例 (其他句子回答「無法判斷」) |
| 至少答對 1 個 | 有回答的句子中，至少有 1 個詞正確的比例 |
| 召回率 | 字幕中的關鍵詞，有多少被模型找到 |

In [ ]:
def outputs(s, threshold):
    """每句輸出的詞 (布林矩陣)：分數 >= 門檻，最多 MAX_WORDS 個。"""
    out = np.zeros_like(s, dtype=bool)
    for i, row in enumerate(s):
        idx = [j for j in np.argsort(-row)[:MAX_WORDS] if row[j] >= threshold]
        out[i, idx] = True
    return out


def evaluate(s, y, threshold, mask=None):
    mask = np.ones(len(s), dtype=bool) if mask is None else mask
    out, y = outputs(s[mask], threshold), y[mask]
    answered = out.any(1)
    hits = (out & (y > 0)).sum()
    return {
        "precision": hits / max(out.sum(), 1),
        "answered": answered.mean(),
        "any_hit": (out & (y > 0)).any(1)[answered].mean() if answered.any() else float("nan"),
        "recall": hits / max(y.sum(), 1),
        "words_per_answer": out.sum() / max(answered.sum(), 1),
    }


def pick_threshold(target):
    for t in np.arange(0.0, 1.0001, 0.005):
        if evaluate(val_scores, val_labels, t)["precision"] >= target:
            return float(t)
    return 1.0


seen = {t for _, t in train_rows}
new = np.array([t not in seen for _, t in test_rows])

print(f"{'驗證集目標':<10}{'門檻':>6}{'測試精確率':>10}{'有回答':>8}{'至少答對1個':>12}{'召回率':>8}{'新句精確率':>10}")
for target in sorted({0.6, 0.65, 0.7, 0.75, 0.8, TARGET_PRECISION}):
    t = pick_threshold(target)
    r, rn = evaluate(test_scores, test_labels, t), evaluate(test_scores, test_labels, t, new)
    mark = "  ← 使用" if target == TARGET_PRECISION else ""
    print(f"{target:<12.0%}{t:>6.3f}{r['precision']:>11.1%}{r['answered']:>9.1%}{r['any_hit']:>12.1%}"
          f"{r['recall']:>9.1%}{rn['precision']:>11.1%}{mark}")

THRESHOLD = pick_threshold(TARGET_PRECISION)
result = evaluate(test_scores, test_labels, THRESHOLD)
print(f"\n使用門檻 {THRESHOLD:.3f}：測試集精確率 {result['precision']:.1%}，"
      f"{result['answered']:.1%} 的句子有回答，平均每次輸出 {result['words_per_answer']:.1f} 個詞")
if result["precision"] < 0.6:
    print("[注意] 測試集精確率低於 60%：把 TARGET_PRECISION 調高再執行一次")

## 2. 存設定檔
`keyword_predict.py` 會讀這個檔案 (門檻、詞彙、模型設定、模型檔位置)。改了上面的設定要重新執行這格。

In [ ]:
config = {
    "keywords": KEYWORDS,
    "threshold": THRESHOLD,
    "max_words": MAX_WORDS,
    "checkpoint": f"{VARIANT}/best.pt",                                   # 相對於 MODEL_DIR
    "norm_file": Path("..", NORM_FILE).as_posix(),                          # 相對於 MODEL_DIR
    "feat_dim": FEAT_DIM,
    "use_face": USE_FACE,
    "frame_stride": FRAME_STRIDE,
    "max_frames": MAX_FRAMES,
    "model": MODEL,
    "expected_test": {k: round(float(v), 4) for k, v in result.items()},
}
(MODEL_DIR / "demo_config.json").write_text(json.dumps(config, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"已存成 {MODEL_DIR / 'demo_config.json'}")

## 3. 檢查推論程式
用 `keyword_predict.py` 重新算 20 句測試資料的分數，跟訓練時存下的分數比較；差距很小才代表推論程式的前處理跟訓練時一致。

In [ ]:
import importlib

import keyword_predict

importlib.reload(keyword_predict)
predictor = keyword_predict.KeywordPredictor(MODEL_DIR / "demo_config.json")

check = np.random.default_rng(0).choice(len(test_rows), min(20, len(test_rows)), replace=False)
diff = max(np.abs(predictor.scores(SEG_DIR / test_rows[i][0]) - test_scores[i]).max() for i in check)
print(f"{len(check)} 句的最大分數差距：{diff:.4f}" + ("  → 一致" if diff < 0.02 else "  → [注意] 差距偏大，請確認設定跟訓練時相同"))

## 4. 示範：輸入一個 .npy
把 `NPY` 換成任何一句的路徑 (Phase5 的 `03segments_face` 或 `03segments` 都可以)。如果這句在資料集裡，會一併顯示字幕來對照。

In [ ]:
NPY = SEG_DIR / test_rows[0][0]          # ← 換成想測試的 .npy 路徑

subtitles = {p: t for p, t in val_rows + test_rows + train_rows}
words = predictor.predict(NPY)
key = Path(NPY).relative_to(SEG_DIR).as_posix() if Path(NPY).is_relative_to(SEG_DIR) else None
text = subtitles.get(key)

print(f"輸入：{NPY}")
if text:
    print(f"字幕：{text}")
if not words:
    print("模型：無法判斷 (沒有任何詞超過門檻)")
else:
    truth = words_of(text) if text else set()
    marks = [f"{w} ({p:.2f}){' ✔' if w in truth else (' ✘' if text else '')}" for w, p in words]
    print("模型：" + "、".join(marks))

## 5. 隨機示範 15 句模型有回答的測試資料

In [ ]:
SHOW = 15
rng = np.random.default_rng(1)
answered = [i for i in rng.permutation(len(test_rows)) if test_scores[i].max() >= THRESHOLD][:SHOW]
n_words = n_hits = 0
for i in answered:
    path, text = test_rows[i]
    truth = words_of(text)
    words = predictor.predict(SEG_DIR / path)
    n_words += len(words)
    n_hits += sum(w in truth for w, _ in words)
    print(f"字幕：{text}\n  模型：" + "、".join(f"{w} ({p:.2f}){' ✔' if w in truth else ' ✘'}" for w, p in words))
print(f"\n這 {len(answered)} 句共輸出 {n_words} 個詞，{n_hits} 個正確 ({n_hits / max(n_words, 1):.0%})")

## 命令列用法
在 `Data Training` 資料夾：
```
python keyword_predict.py "..\Data Preparation\Phase5-Pause\output\03segments_face\2\10\000641_001738_s00.npy"
```
想讓模型多回答一些句子，就把 `TARGET_PRECISION` 調低 (準確率會跟著下降)；想更準就調高 (回答的句子會變少)，再重新執行第 1、2 步。